# 🧠 智腦記事本 — Google Colab 版
## AI 可搜尋、自我精進的筆記系統（Gemini + ChromaDB + Mermaid 知識地圖）

### 使用步驟：
1. **依序執行每個儲存格**（Ctrl+Enter 或點左側 ▶ 按鈕）
2. 在「設定 API 金鑰」儲存格輸入你的 Google AI API 金鑰
3. 最後儲存格執行完畢後，**點擊輸出的網址**即可開啟應用程式

> **取得 Google AI API 金鑰**：前往 https://aistudio.google.com/apikey 免費申請


In [ ]:
# ① 安裝所需套件（約 1-2 分鐘）
print("📦 安裝套件中，請稍候...")
import subprocess, sys

pkgs = ["google-genai", "flask", "chromadb", "flask-cors"]
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs,
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ 套件安裝完成！")
else:
    print("⚠️ 安裝時有警告：", result.stderr[-500:] if result.stderr else "")
    print("✅ 繼續執行...")


In [ ]:
# ② 設定 Google AI API 金鑰
# 方法 A：直接填入（較方便，但金鑰會顯示在筆記本中）
# 方法 B：使用 Colab Secrets（左側鑰匙圖示 🔑 → 新增 GOOGLE_API_KEY）

import os

# ── 方法 B：從 Colab Secrets 讀取（推薦）────────────────────
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    if GOOGLE_API_KEY:
        print("✅ 已從 Colab Secrets 讀取 API 金鑰")
    else:
        raise ValueError("Secrets 中沒有 GOOGLE_API_KEY")
except Exception as e:
    # ── 方法 A：直接填入 ────────────────────────────────────
    GOOGLE_API_KEY = ""  # ← 在這裡填入你的 API 金鑰
    if not GOOGLE_API_KEY:
        print("⚠️  請填入 API 金鑰！")
        print("   方法 A：在上方 GOOGLE_API_KEY = '' 引號中填入金鑰")
        print("   方法 B：點左側鑰匙圖示 🔑 新增名稱為 GOOGLE_API_KEY 的 Secret")
    else:
        print("✅ 已從直接填入讀取 API 金鑰")

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
print(f"金鑰狀態：{'✓ 已設定（長度 ' + str(len(GOOGLE_API_KEY)) + '）' if GOOGLE_API_KEY else '✗ 未設定'}")


In [ ]:
# ③ 掛載 Google Drive（可選）
# 執行此儲存格可將筆記永久保存到 Google Drive
# 若不需要持久化，可直接跳過此儲存格

USE_DRIVE = False  # ← 改為 True 並執行，啟用 Google Drive 儲存

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/智腦記事本'
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"✅ Google Drive 已掛載，資料將儲存至：{DATA_DIR}")
else:
    DATA_DIR = '/content'
    print(f"📁 使用暫時儲存（重啟後資料會消失）：{DATA_DIR}")
    print("   如需持久化，請將 USE_DRIVE 改為 True")

os.environ['NOTEBOOK_DIR'] = DATA_DIR
print(f"資料目錄：{DATA_DIR}")


In [ ]:
# ④ 寫入應用程式核心檔案
import os

app_code = """#!/usr/bin/env python3
\"\"\"
智腦記事本 — 網頁介面 (Google GenAI + ChromaDB 版)
使用方式：python app.py → 瀏覽器開啟 http://localhost:5000

需要環境變數：GOOGLE_API_KEY
（可在專案目錄建立 .env 檔案自動載入）
\"\"\"

from flask import Flask, request, jsonify, render_template, Response, stream_with_context
import sqlite3
import json
import os
import re
from datetime import datetime

from google import genai
from google.genai import types
import chromadb

app = Flask(__name__)

BASE_DIR = os.environ.get('NOTEBOOK_DIR', '/content')
DB_PATH = os.path.join(BASE_DIR, "notebook.db")
CHROMA_PATH = os.path.join(BASE_DIR, "chroma_data")

# ---------------------------------------------------------------------------
# Model names — adjust if your API key has different access
# ---------------------------------------------------------------------------
TEXT_MODEL  = "gemini-2.0-flash"          # fast & capable; change to gemini-1.5-pro if preferred
EMBED_MODEL = "text-embedding-004"         # stable embedding model

# ---------------------------------------------------------------------------
# Google GenAI client (singleton)
# ---------------------------------------------------------------------------

_genai_client: genai.Client | None = None

def get_client() -> genai.Client:
    global _genai_client
    if _genai_client is None:
        api_key = os.environ.get("GOOGLE_API_KEY")
        if not api_key:
            raise RuntimeError("未設定 GOOGLE_API_KEY，請建立 .env 檔案或設定環境變數")
        _genai_client = genai.Client(api_key=api_key)
    return _genai_client


def embed_text(text: str) -> list[float]:
    \"\"\"Generate embedding vector via Google text-embedding model.\"\"\"
    client = get_client()
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=text[:8000],
    )
    return result.embeddings[0].values


def stream_gemini(prompt: str):
    \"\"\"Generator yielding text chunks from Gemini streaming response.\"\"\"
    client = get_client()
    for chunk in client.models.generate_content_stream(
        model=TEXT_MODEL,
        contents=prompt,
    ):
        text = chunk.text
        if text:
            yield text


# ---------------------------------------------------------------------------
# ChromaDB (semantic vector store)
# ---------------------------------------------------------------------------

def get_chroma() -> chromadb.Collection:
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    return client.get_or_create_collection(
        name="notes",
        metadata={"hnsw:space": "cosine"},
    )


def upsert_embedding(note_id: int, title: str, content: str, tags: list[str]) -> None:
    \"\"\"Upsert a note's embedding into ChromaDB (best-effort).\"\"\"
    try:
        col = get_chroma()
        doc = f"{title}\\n\\n{content}\\n\\nTags: {', '.join(tags)}"
        vec = embed_text(doc)
        col.upsert(
            ids=[str(note_id)],
            documents=[doc],
            embeddings=[vec],
            metadatas=[{"title": title, "note_id": note_id}],
        )
    except Exception as e:
        print(f"[embedding] upsert 失敗 (note #{note_id}): {e}")


def delete_embedding(note_id: int) -> None:
    try:
        get_chroma().delete(ids=[str(note_id)])
    except Exception:
        pass


def semantic_search(query: str, n_results: int = 10) -> list[dict]:
    \"\"\"Vector similarity search via ChromaDB.\"\"\"
    col = get_chroma()
    if col.count() == 0:
        return []
    try:
        qvec = embed_text(query)
        results = col.query(
            query_embeddings=[qvec],
            n_results=min(n_results, col.count()),
        )
        out = []
        for idx, mid in enumerate(results["ids"][0]):
            out.append({
                "note_id": int(mid),
                "distance": results["distances"][0][idx] if results.get("distances") else 0,
            })
        return out
    except Exception as e:
        print(f"[semantic_search] 失敗: {e}")
        return []


# ---------------------------------------------------------------------------
# SQLite Database
# ---------------------------------------------------------------------------

def init_db() -> None:
    conn = sqlite3.connect(DB_PATH)
    conn.execute(\"\"\"
        CREATE TABLE IF NOT EXISTS notes (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            title       TEXT    NOT NULL,
            content     TEXT    NOT NULL,
            tags        TEXT    DEFAULT '[]',
            created_at  TEXT    NOT NULL,
            updated_at  TEXT    NOT NULL
        )
    \"\"\")
    conn.execute(\"\"\"
        CREATE VIRTUAL TABLE IF NOT EXISTS notes_fts USING fts5(
            title, content, tags,
            content='notes', content_rowid='id'
        )
    \"\"\")
    conn.executescript(\"\"\"
        CREATE TRIGGER IF NOT EXISTS notes_ai AFTER INSERT ON notes BEGIN
            INSERT INTO notes_fts(rowid, title, content, tags)
            VALUES (new.id, new.title, new.content, new.tags);
        END;
        CREATE TRIGGER IF NOT EXISTS notes_ad AFTER DELETE ON notes BEGIN
            INSERT INTO notes_fts(notes_fts, rowid, title, content, tags)
            VALUES ('delete', old.id, old.title, old.content, old.tags);
        END;
        CREATE TRIGGER IF NOT EXISTS notes_au AFTER UPDATE ON notes BEGIN
            INSERT INTO notes_fts(notes_fts, rowid, title, content, tags)
            VALUES ('delete', old.id, old.title, old.content, old.tags);
            INSERT INTO notes_fts(rowid, title, content, tags)
            VALUES (new.id, new.title, new.content, new.tags);
        END;
    \"\"\")
    conn.commit()
    conn.close()


def _row_to_dict(row, cols) -> dict:
    n = dict(zip(cols, row))
    if "tags" in n:
        n["tags"] = json.loads(n["tags"])
    return n

_NOTE_COLS = ["id", "title", "content", "tags", "created_at", "updated_at"]


def db_get_note(note_id: int) -> dict | None:
    conn = sqlite3.connect(DB_PATH)
    row = conn.execute(
        "SELECT id, title, content, tags, created_at, updated_at FROM notes WHERE id = ?",
        (note_id,),
    ).fetchone()
    conn.close()
    return _row_to_dict(row, _NOTE_COLS) if row else None


def db_all_notes() -> list[dict]:
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(
        "SELECT id, title, content, tags, created_at, updated_at FROM notes ORDER BY updated_at DESC"
    ).fetchall()
    conn.close()
    return [_row_to_dict(r, _NOTE_COLS) for r in rows]


def db_fts_search(query: str, limit: int = 20) -> list[dict]:
    conn = sqlite3.connect(DB_PATH)
    cols = ["id", "title", "content", "tags", "created_at", "snippet"]
    try:
        rows = conn.execute(\"\"\"
            SELECT n.id, n.title, n.content, n.tags, n.created_at,
                   snippet(notes_fts, 1, '**', '**', '...', 30) AS snippet
            FROM notes_fts
            JOIN notes n ON n.id = notes_fts.rowid
            WHERE notes_fts MATCH ?
            ORDER BY rank LIMIT ?
        \"\"\", (query, limit)).fetchall()
    except Exception:
        like = f"%{query}%"
        rows = conn.execute(\"\"\"
            SELECT id, title, content, tags, created_at, SUBSTR(content,1,200) AS snippet
            FROM notes WHERE title LIKE ? OR content LIKE ? LIMIT ?
        \"\"\", (like, like, limit)).fetchall()
    conn.close()
    return [_row_to_dict(r, cols) for r in rows]


def db_save_note(title: str, content: str, tags: list | None = None) -> int:
    now = datetime.now().isoformat()
    conn = sqlite3.connect(DB_PATH)
    cur = conn.execute(
        "INSERT INTO notes (title, content, tags, created_at, updated_at) VALUES (?,?,?,?,?)",
        (title, content, json.dumps(tags or []), now, now),
    )
    nid = cur.lastrowid
    conn.commit()
    conn.close()
    return nid


def db_update_note(note_id: int, content: str = None, tags: list = None) -> None:
    conn = sqlite3.connect(DB_PATH)
    now = datetime.now().isoformat()
    if content is not None and tags is not None:
        conn.execute("UPDATE notes SET content=?, tags=?, updated_at=? WHERE id=?",
                     (content, json.dumps(tags), now, note_id))
    elif content is not None:
        conn.execute("UPDATE notes SET content=?, updated_at=? WHERE id=?",
                     (content, now, note_id))
    elif tags is not None:
        conn.execute("UPDATE notes SET tags=?, updated_at=? WHERE id=?",
                     (json.dumps(tags), now, note_id))
    conn.commit()
    conn.close()


def db_delete_note(note_id: int) -> bool:
    conn = sqlite3.connect(DB_PATH)
    affected = conn.execute("DELETE FROM notes WHERE id=?", (note_id,)).rowcount
    conn.commit()
    conn.close()
    return affected > 0


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def sse(data: dict) -> str:
    return f"data: {json.dumps(data, ensure_ascii=False)}\\n\\n"


def _sse_headers():
    return {"X-Accel-Buffering": "no", "Cache-Control": "no-cache"}


def run_autotag(note_id: int, title: str, content: str) -> list[str]:
    try:
        client = get_client()
        response = client.models.generate_content(
            model=TEXT_MODEL,
            contents=(
                f"為以下筆記產生 3-7 個相關標籤。只輸出 JSON 陣列，不要其他文字。\\n"
                f"範例：[\\"python\\", \\"程式設計\\", \\"教學\\"]\\n\\n"
                f"標題：{title}\\n內容：{content[:400]}"
            ),
        )
        text = response.text.strip()
        match = re.search(r"\\[.*?\\]", text, re.DOTALL)
        if match:
            tags = json.loads(match.group())
            db_update_note(note_id, tags=tags)
            return tags
    except Exception as e:
        print(f"[autotag] 失敗: {e}")
    return []


# ---------------------------------------------------------------------------
# Routes — pages
# ---------------------------------------------------------------------------

@app.route("/")
def index():
    return render_template("index.html")


@app.route("/api/status")
def api_status():
    \"\"\"Check if API key is configured.\"\"\"
    api_key = os.environ.get("GOOGLE_API_KEY")
    return jsonify({
        "api_key_set": bool(api_key),
        "text_model": TEXT_MODEL,
        "embed_model": EMBED_MODEL,
    })


# ---------------------------------------------------------------------------
# Routes — REST API
# ---------------------------------------------------------------------------

@app.route("/api/notes", methods=["GET"])
def api_list_notes():
    return jsonify(db_all_notes())


@app.route("/api/notes", methods=["POST"])
def api_add_note():
    data = request.get_json()
    title = (data.get("title") or "").strip()
    content = (data.get("content") or "").strip()
    if not title or not content:
        return jsonify({"error": "標題和內容不能為空"}), 400
    note_id = db_save_note(title, content)
    tags = run_autotag(note_id, title, content)
    upsert_embedding(note_id, title, content, tags)
    note = db_get_note(note_id)
    return jsonify(note), 201


@app.route("/api/notes/<int:note_id>", methods=["GET"])
def api_get_note(note_id):
    note = db_get_note(note_id)
    if not note:
        return jsonify({"error": "找不到筆記"}), 404
    return jsonify(note)


@app.route("/api/notes/<int:note_id>", methods=["DELETE"])
def api_delete_note(note_id):
    if not db_delete_note(note_id):
        return jsonify({"error": "找不到筆記"}), 404
    delete_embedding(note_id)
    return jsonify({"ok": True})


@app.route("/api/notes/<int:note_id>/autotag", methods=["POST"])
def api_autotag(note_id):
    note = db_get_note(note_id)
    if not note:
        return jsonify({"error": "找不到筆記"}), 404
    tags = run_autotag(note_id, note["title"], note["content"])
    upsert_embedding(note_id, note["title"], note["content"], tags)
    return jsonify({"tags": tags})


# ---------------------------------------------------------------------------
# Routes — AI streaming (SSE)
# ---------------------------------------------------------------------------

@app.route("/api/search", methods=["POST"])
def api_search():
    data = request.get_json()
    query = (data.get("query") or "").strip()
    if not query:
        return jsonify({"error": "請輸入搜尋詞"}), 400

    sem_hits = semantic_search(query, n_results=15)
    fts_results = db_fts_search(query, limit=15)

    all_ids = list(dict.fromkeys([h["note_id"] for h in sem_hits] +
                                 [r["id"] for r in fts_results]))
    notes_map = {n["id"]: n for n in db_all_notes()}

    pool = [notes_map[nid] for nid in all_ids if nid in notes_map]
    if not pool:
        pool = list(notes_map.values())[:50]

    def generate():
        if not pool:
            yield sse({"type": "text", "content": "記事本是空的，請先新增筆記。"})
            yield sse({"type": "done", "results": []})
            return

        overview = "\\n".join(
            f"ID {r['id']}: {r['title']} — {r['content'][:100]}..."
            for r in pool
        )
        prompt = (
            f"你是一個智能筆記搜尋助手。用戶的搜尋關鍵字是：「{query}」\\n\\n"
            f"以下是根據語意向量排序的相關筆記：\\n{overview}\\n\\n"
            f"請根據語意相關性排列筆記，說明每篇筆記與搜尋詞的關聯。\\n"
            f"- 列出最相關的筆記（格式：**#ID 標題** — 原因）\\n"
            f"- 最後給一句話總結\\n"
            f"- 若都不相關，請如實說明"
        )
        try:
            for text in stream_gemini(prompt):
                yield sse({"type": "text", "content": text})
        except Exception as e:
            yield sse({"type": "error", "message": str(e)})
            return

        yield sse({"type": "done", "results": [
            {"id": r["id"], "title": r["title"],
             "snippet": r["content"][:120], "tags": r["tags"]}
            for r in pool[:8]
        ]})

    return Response(stream_with_context(generate()),
                    mimetype="text/event-stream", headers=_sse_headers())


@app.route("/api/notes/<int:note_id>/augment", methods=["POST"])
def api_augment(note_id):
    note = db_get_note(note_id)
    if not note:
        return jsonify({"error": "找不到筆記"}), 404

    def generate():
        prompt = (
            f"你是一個知識補充系統。請根據以下筆記的主題，補充更多相關知識：\\n\\n"
            f"**筆記標題**：{note['title']}\\n"
            f"**筆記內容**：\\n{note['content']}\\n\\n"
            f"請補充：\\n"
            f"1. 相關概念與延伸說明\\n"
            f"2. 可能遺漏的重要細節\\n"
            f"3. 實際應用範例或使用場景\\n"
            f"4. 與其他相關主題的連結\\n"
            f"5. 值得深入了解的延伸方向\\n\\n"
            f"請以清晰有結構的方式撰寫（使用與筆記相同的語言）。"
        )
        buf = []
        try:
            for text in stream_gemini(prompt):
                buf.append(text)
                yield sse({"type": "text", "content": text})
        except Exception as e:
            yield sse({"type": "error", "message": str(e)})
            return

        supplement = "".join(buf)
        new_content = note["content"] + "\\n\\n---\\n**🤖 AI 補充知識：**\\n\\n" + supplement
        db_update_note(note_id, content=new_content)
        upsert_embedding(note_id, note["title"], new_content, note["tags"])
        yield sse({"type": "done", "message": "已補充並儲存至筆記", "note_id": note_id})

    return Response(stream_with_context(generate()),
                    mimetype="text/event-stream", headers=_sse_headers())


@app.route("/api/notes/<int:note_id>/improve", methods=["POST"])
def api_improve(note_id):
    note = db_get_note(note_id)
    if not note:
        return jsonify({"error": "找不到筆記"}), 404

    def generate():
        prompt = (
            f"請改善以下筆記的品質，包括：\\n"
            f"1. 改善清晰度與可讀性\\n"
            f"2. 加入更好的結構（標題、條列式）\\n"
            f"3. 修正任何不精確的說法\\n"
            f"4. 保留所有重要資訊，並使整體更簡潔有力\\n\\n"
            f"**標題**：{note['title']}\\n"
            f"**原始內容**：\\n{note['content']}\\n\\n"
            f"請直接輸出改善後的筆記內容（使用與原文相同的語言），不需要額外說明。"
        )
        buf = []
        try:
            for text in stream_gemini(prompt):
                buf.append(text)
                yield sse({"type": "text", "content": text})
        except Exception as e:
            yield sse({"type": "error", "message": str(e)})
            return

        improved = "".join(buf)
        db_update_note(note_id, content=improved)
        upsert_embedding(note_id, note["title"], improved, note["tags"])
        yield sse({"type": "done", "message": "已改善並儲存至筆記", "note_id": note_id})

    return Response(stream_with_context(generate()),
                    mimetype="text/event-stream", headers=_sse_headers())


@app.route("/api/analyze", methods=["GET"])
def api_analyze():
    notes = db_all_notes()

    def generate():
        if not notes:
            yield sse({"type": "text", "content": "記事本是空的，請先新增筆記。"})
            yield sse({"type": "done"})
            return

        overview = "\\n".join(
            f"#{n['id']}: {n['title']} | 標籤: {', '.join(n['tags']) or '無'}"
            for n in notes
        )
        prompt = (
            f"你是一位知識管理顧問。請分析以下記事本的知識結構：\\n\\n"
            f"**目前的筆記：**\\n{overview}\\n\\n"
            f"請提供：\\n"
            f"1. **知識缺口分析**：哪些重要主題或關聯性缺失？\\n"
            f"2. **建議新增筆記**：具體建議 3-5 個應該新增的主題\\n"
            f"3. **知識連結地圖**：現有筆記彼此之間的關聯性\\n"
            f"4. **優先改善建議**：哪 2-3 篇筆記最需要擴充或改善？\\n\\n"
            f"請給出具體且可執行的建議。"
        )
        try:
            for text in stream_gemini(prompt):
                yield sse({"type": "text", "content": text})
        except Exception as e:
            yield sse({"type": "error", "message": str(e)})
            return

        yield sse({"type": "done"})

    return Response(stream_with_context(generate()),
                    mimetype="text/event-stream", headers=_sse_headers())


@app.route("/api/summarize", methods=["POST"])
def api_summarize():
    data = request.get_json()
    topic = (data.get("topic") or "").strip()
    if not topic:
        return jsonify({"error": "請輸入主題"}), 400

    sem_hits = semantic_search(topic, n_results=10)
    fts_results = db_fts_search(topic, limit=10)

    all_ids = list(dict.fromkeys(
        [h["note_id"] for h in sem_hits] + [r["id"] for r in fts_results]
    ))
    notes_map = {n["id"]: n for n in db_all_notes()}
    results = [notes_map[nid] for nid in all_ids if nid in notes_map]

    def generate():
        if not results:
            yield sse({"type": "text", "content": f"找不到與「{topic}」相關的筆記。"})
            yield sse({"type": "done"})
            return

        notes_content = "\\n\\n---\\n\\n".join(
            f"**#{r['id']} {r['title']}**\\n{r['content']}"
            for r in results
        )
        prompt = (
            f"請根據以下筆記，針對主題「{topic}」合成一份全面的知識摘要：\\n\\n"
            f"{notes_content}\\n\\n"
            f"請包含：\\n"
            f"1. 主題核心概念的清晰摘要\\n"
            f"2. 整合各筆記的關鍵洞見\\n"
            f"3. 不同觀點或方法的對比\\n"
            f"4. 知識的實際應用\\n"
            f"5. 尚待補充的知識空白\\n\\n"
            f"格式要清晰有層次。"
        )
        try:
            for text in stream_gemini(prompt):
                yield sse({"type": "text", "content": text})
        except Exception as e:
            yield sse({"type": "error", "message": str(e)})
            return

        yield sse({"type": "done"})

    return Response(stream_with_context(generate()),
                    mimetype="text/event-stream", headers=_sse_headers())


# ---------------------------------------------------------------------------
# Routes — Mind Map (知識地圖)
# ---------------------------------------------------------------------------

@app.route("/api/mindmap", methods=["GET"])
def api_mindmap():
    notes = db_all_notes()

    def generate():
        if not notes:
            yield sse({"type": "text", "content": "記事本是空的，請先新增筆記。"})
            yield sse({"type": "done"})
            return

        overview = "\\n".join(
            f"#{n['id']}: {n['title']}\\n   內容摘要: {n['content'][:150]}...\\n   標籤: {', '.join(n['tags']) or '無'}"
            for n in notes
        )

        analysis_prompt = (
            f"你是一位知識架構師。請分析以下所有筆記，產出一份「知識地圖」報告：\\n\\n"
            f"{overview}\\n\\n"
            f"請完成以下工作：\\n"
            f"1. **主題群組**：將筆記分成幾個知識領域/群組\\n"
            f"2. **連結關係**：說明筆記之間的關聯（哪些筆記共享概念、互相補充或依賴）\\n"
            f"3. **核心節點**：哪些筆記是知識網路中最重要的樞紐？\\n"
            f"4. **擴展建議**：基於現有知識地圖，建議下一步可以新增的 3 個知識節點\\n\\n"
            f"最後，請產生一段 Mermaid mindmap 圖表語法來視覺化這個知識地圖。\\n"
            f"語法範例：\\n"
            f"```mermaid\\n"
            f"mindmap\\n"
            f"  root((我的知識庫))\\n"
            f"    程式設計\\n"
            f"      Python\\n"
            f"      Flask\\n"
            f"    資料科學\\n"
            f"      機器學習\\n"
            f"```\\n"
            f"請用繁體中文，確保 Mermaid 語法正確可渲染。"
        )

        buf = []
        try:
            for text in stream_gemini(analysis_prompt):
                buf.append(text)
                yield sse({"type": "text", "content": text})
        except Exception as e:
            yield sse({"type": "error", "message": str(e)})
            return

        full = "".join(buf)
        match = re.search(r"```mermaid\\s*\\n(.*?)```", full, re.DOTALL)
        if match:
            yield sse({"type": "mermaid", "content": match.group(1).strip()})

        yield sse({"type": "done"})

    return Response(stream_with_context(generate()),
                    mimetype="text/event-stream", headers=_sse_headers())


# ---------------------------------------------------------------------------
# Startup
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    init_db()
    port = int(os.environ.get("PORT", 5000))
    api_key = os.environ.get("GOOGLE_API_KEY")
    print(f"\\n🧠 智腦記事本 (Google GenAI + ChromaDB) 已啟動！")
    print(f"   📡 文字模型：{TEXT_MODEL}")
    print(f"   🔢 嵌入模型：{EMBED_MODEL}")
    print(f"   🔑 API 金鑰：{'✓ 已設定' if api_key else '✗ 未設定（請建立 .env 或設定 GOOGLE_API_KEY）'}")
    print(f"   🌐 網址：http://localhost:{port}\\n")
    app.run(host="0.0.0.0", debug=False, use_reloader=False, threaded=True, port=port)
"""

with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)
print("✅ app.py 寫入完成")


In [ ]:
# ⑤ 寫入網頁介面
import os

os.makedirs("/content/templates", exist_ok=True)

html_code = """<!DOCTYPE html>
<html lang="zh-TW">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>🧠 智腦記事本</title>
<script src="https://cdn.jsdelivr.net/npm/marked/marked.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.min.js"></script>
<style>
  /* ── Reset & base ───────────────────────────────── */
  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    background: #f1f5f9;
    color: #0f172a;
    height: 100vh;
    display: flex;
    flex-direction: column;
    overflow: hidden;
  }

  /* ── Header ─────────────────────────────────────── */
  .header {
    background: #1e293b;
    color: #f8fafc;
    padding: 0 20px;
    height: 56px;
    display: flex;
    align-items: center;
    gap: 12px;
    flex-shrink: 0;
    box-shadow: 0 2px 8px rgba(0,0,0,.25);
    z-index: 10;
  }
  .header-logo { font-size: 1.25rem; font-weight: 700; white-space: nowrap; }
  .header-search {
    flex: 1;
    display: flex;
    gap: 8px;
    max-width: 560px;
    margin: 0 auto;
  }
  .header-search input {
    flex: 1;
    padding: 7px 14px;
    border-radius: 8px;
    border: none;
    background: #334155;
    color: #f8fafc;
    font-size: .9rem;
    outline: none;
  }
  .header-search input::placeholder { color: #94a3b8; }
  .header-search input:focus { background: #3e5270; }
  .header-actions { display: flex; gap: 8px; }

  /* ── Buttons ─────────────────────────────────────── */
  .btn {
    padding: 7px 14px;
    border: none;
    border-radius: 8px;
    cursor: pointer;
    font-size: .85rem;
    font-weight: 600;
    transition: opacity .15s, transform .1s;
    display: inline-flex;
    align-items: center;
    gap: 5px;
    white-space: nowrap;
  }
  .btn:active { transform: scale(.97); }
  .btn:disabled { opacity: .5; cursor: not-allowed; }
  .btn-primary   { background: #4f46e5; color: #fff; }
  .btn-primary:hover:not(:disabled)  { background: #4338ca; }
  .btn-violet    { background: #7c3aed; color: #fff; }
  .btn-violet:hover:not(:disabled)   { background: #6d28d9; }
  .btn-teal      { background: #0d9488; color: #fff; }
  .btn-teal:hover:not(:disabled)     { background: #0f766e; }
  .btn-amber     { background: #d97706; color: #fff; }
  .btn-amber:hover:not(:disabled)    { background: #b45309; }
  .btn-rose      { background: #e11d48; color: #fff; }
  .btn-rose:hover:not(:disabled)     { background: #be123c; }
  .btn-slate     { background: #475569; color: #fff; }
  .btn-slate:hover:not(:disabled)    { background: #334155; }
  .btn-ghost     { background: #e2e8f0; color: #374151; }
  .btn-ghost:hover:not(:disabled)    { background: #cbd5e1; }
  .btn-emerald   { background: #059669; color: #fff; }
  .btn-emerald:hover:not(:disabled)  { background: #047857; }
  .btn-sm        { padding: 5px 10px; font-size: .8rem; }

  /* ── Layout ──────────────────────────────────────── */
  .layout { display: flex; flex: 1; overflow: hidden; }

  /* ── Sidebar ─────────────────────────────────────── */
  .sidebar {
    width: 260px; min-width: 260px;
    background: #1e293b; color: #e2e8f0;
    display: flex; flex-direction: column;
    overflow: hidden; border-right: 1px solid #0f172a;
  }
  .sidebar-top {
    padding: 12px;
    display: flex; flex-direction: column; gap: 8px;
    border-bottom: 1px solid #334155;
  }
  .sidebar-top .btn { width: 100%; justify-content: center; }
  .notes-list { flex: 1; overflow-y: auto; padding: 8px 0; }
  .notes-list::-webkit-scrollbar { width: 4px; }
  .notes-list::-webkit-scrollbar-thumb { background: #475569; border-radius: 4px; }
  .note-item {
    padding: 10px 14px; cursor: pointer;
    border-left: 3px solid transparent; transition: background .15s;
  }
  .note-item:hover { background: #334155; }
  .note-item.active { background: #334155; border-left-color: #6366f1; }
  .note-item-title {
    font-size: .88rem; font-weight: 600; color: #f1f5f9;
    white-space: nowrap; overflow: hidden; text-overflow: ellipsis;
  }
  .note-item-meta { font-size: .75rem; color: #94a3b8; margin-top: 3px; }
  .note-item-tags { display: flex; gap: 4px; flex-wrap: wrap; margin-top: 4px; }
  .tag { background: #312e81; color: #a5b4fc; font-size: .7rem; padding: 1px 6px; border-radius: 100px; }
  .sidebar-bottom {
    padding: 12px; border-top: 1px solid #334155;
    display: flex; flex-direction: column; gap: 6px;
  }
  .sidebar-bottom .btn { width: 100%; justify-content: center; }
  .note-count { text-align: center; font-size: .75rem; color: #64748b; padding: 4px 0; }

  /* ── Main content ────────────────────────────────── */
  .main { flex: 1; display: flex; flex-direction: column; overflow: hidden; background: #f1f5f9; }

  /* ── Note view ───────────────────────────────────── */
  .note-view { flex: 1; overflow-y: auto; padding: 28px 32px; display: none; }
  .note-view.visible { display: block; }
  .note-view::-webkit-scrollbar { width: 6px; }
  .note-view::-webkit-scrollbar-thumb { background: #cbd5e1; border-radius: 4px; }
  .note-header { display: flex; align-items: flex-start; justify-content: space-between; gap: 16px; margin-bottom: 16px; }
  .note-title { font-size: 1.6rem; font-weight: 700; color: #0f172a; line-height: 1.3; }
  .note-actions { display: flex; gap: 8px; flex-wrap: wrap; flex-shrink: 0; }
  .note-tags { display: flex; gap: 6px; flex-wrap: wrap; margin-bottom: 12px; }
  .note-tag { background: #ddd6fe; color: #5b21b6; font-size: .78rem; padding: 2px 10px; border-radius: 100px; font-weight: 500; }
  .note-meta { font-size: .78rem; color: #94a3b8; margin-bottom: 20px; }

  /* shared rich-text styles */
  .rich-text { line-height: 1.7; font-size: .95rem; color: #1e293b; }
  .rich-text h1, .rich-text h2, .rich-text h3 { margin: 1em 0 .5em; color: #0f172a; }
  .rich-text p { margin-bottom: .8em; }
  .rich-text ul, .rich-text ol { padding-left: 1.5em; margin-bottom: .8em; }
  .rich-text li { margin-bottom: .3em; }
  .rich-text code { background: #f1f5f9; padding: 1px 5px; border-radius: 4px; font-size: .88em; }
  .rich-text pre { background: #1e293b; color: #e2e8f0; padding: 14px; border-radius: 8px; overflow-x: auto; margin-bottom: .8em; }
  .rich-text pre code { background: none; padding: 0; }
  .rich-text strong { font-weight: 700; }
  .rich-text hr { border: none; border-top: 1px solid #e2e8f0; margin: 1.5em 0; }
  .rich-text blockquote { border-left: 3px solid #6366f1; padding-left: 14px; color: #64748b; margin: .8em 0; }

  .note-body {
    background: #fff; border-radius: 12px; padding: 24px 28px;
    box-shadow: 0 1px 4px rgba(0,0,0,.06);
  }

  /* ── AI output panel ─────────────────────────────── */
  .ai-panel { flex: 1; display: none; flex-direction: column; overflow: hidden; }
  .ai-panel.visible { display: flex; }
  .ai-panel-header {
    padding: 16px 28px 12px; background: #f1f5f9;
    border-bottom: 1px solid #e2e8f0;
    display: flex; align-items: center; gap: 12px;
  }
  .ai-panel-title { font-weight: 700; font-size: 1rem; color: #1e293b; flex: 1; }
  .ai-stream-area { flex: 1; overflow-y: auto; padding: 20px 32px 24px; }
  .ai-stream-area::-webkit-scrollbar { width: 6px; }
  .ai-stream-area::-webkit-scrollbar-thumb { background: #cbd5e1; border-radius: 4px; }
  .ai-bubble {
    background: #fff; border-radius: 12px; padding: 20px 24px;
    box-shadow: 0 1px 4px rgba(0,0,0,.06); border: 1px solid #e2e8f0;
    min-height: 80px;
  }
  .ai-results { margin-top: 16px; display: flex; flex-direction: column; gap: 8px; }
  .ai-result-item {
    background: #eff6ff; border: 1px solid #bfdbfe; border-radius: 10px;
    padding: 10px 16px; cursor: pointer; transition: background .15s;
  }
  .ai-result-item:hover { background: #dbeafe; }
  .ai-result-item-title { font-weight: 600; font-size: .9rem; color: #1d4ed8; }
  .ai-result-item-snippet { font-size: .82rem; color: #475569; margin-top: 3px; }
  .status-bar {
    padding: 8px 28px; font-size: .8rem; color: #64748b;
    background: #f8fafc; border-top: 1px solid #e2e8f0;
    display: flex; align-items: center; gap: 8px; min-height: 34px;
  }
  .spinner {
    width: 14px; height: 14px;
    border: 2px solid #cbd5e1; border-top-color: #6366f1;
    border-radius: 50%; animation: spin .7s linear infinite; flex-shrink: 0;
  }
  @keyframes spin { to { transform: rotate(360deg); } }

  /* ── Mermaid diagram ─────────────────────────────── */
  .mermaid-container {
    margin-top: 20px;
    background: #fff;
    border-radius: 12px;
    padding: 24px;
    box-shadow: 0 1px 4px rgba(0,0,0,.06);
    border: 1px solid #e2e8f0;
    overflow-x: auto;
  }
  .mermaid-container svg { max-width: 100%; height: auto; }

  /* ── Empty state ─────────────────────────────────── */
  .empty-state {
    flex: 1; display: flex; flex-direction: column;
    align-items: center; justify-content: center; color: #94a3b8; gap: 12px;
  }
  .empty-state.hidden { display: none; }
  .empty-state-icon { font-size: 3rem; }
  .empty-state-text { font-size: 1rem; }

  /* ── Modal ───────────────────────────────────────── */
  .modal-overlay {
    display: none; position: fixed; inset: 0;
    background: rgba(0,0,0,.5); z-index: 100;
    align-items: center; justify-content: center;
  }
  .modal-overlay.open { display: flex; }
  .modal {
    background: #fff; border-radius: 14px; padding: 28px;
    width: 540px; max-width: 95vw;
    box-shadow: 0 20px 60px rgba(0,0,0,.2);
    display: flex; flex-direction: column; gap: 16px;
  }
  .modal-title { font-size: 1.1rem; font-weight: 700; color: #0f172a; }
  .form-group { display: flex; flex-direction: column; gap: 6px; }
  .form-label { font-size: .85rem; font-weight: 600; color: #374151; }
  .form-input {
    padding: 9px 13px; border: 1.5px solid #e2e8f0; border-radius: 8px;
    font-size: .9rem; color: #0f172a; outline: none;
    transition: border-color .15s; font-family: inherit;
  }
  .form-input:focus { border-color: #6366f1; }
  textarea.form-input { resize: vertical; min-height: 120px; }
  .modal-actions { display: flex; justify-content: flex-end; gap: 8px; margin-top: 4px; }

  /* ── Summarize bar ───────────────────────────────── */
  .summarize-bar {
    display: none; align-items: center; gap: 8px;
    padding: 10px 28px; background: #fff7ed; border-bottom: 1px solid #fed7aa;
  }
  .summarize-bar.visible { display: flex; }
  .summarize-bar input {
    flex: 1; padding: 7px 12px; border: 1.5px solid #fb923c;
    border-radius: 8px; font-size: .9rem; outline: none;
    font-family: inherit; color: #0f172a;
  }
  .summarize-bar input:focus { border-color: #ea580c; }

  /* ── Toast ───────────────────────────────────────── */
  .toast {
    position: fixed; bottom: 24px; right: 24px;
    background: #1e293b; color: #f8fafc;
    padding: 12px 20px; border-radius: 10px;
    font-size: .88rem; font-weight: 500;
    box-shadow: 0 4px 20px rgba(0,0,0,.25);
    z-index: 200; opacity: 0; transform: translateY(10px);
    transition: opacity .25s, transform .25s; max-width: 320px;
  }
  .toast.show { opacity: 1; transform: translateY(0); }
  .toast.success { border-left: 4px solid #10b981; }
  .toast.error   { border-left: 4px solid #e11d48; }
</style>
</head>
<body>

<!-- Header -->
<header class="header">
  <div class="header-logo">🧠 智腦記事本</div>
  <div class="header-search">
    <input id="searchInput" type="text" placeholder="語意搜尋…（AI 理解你的意思，不只是關鍵字）"
      onkeydown="if(event.key==='Enter') doSearch()">
    <button class="btn btn-primary" onclick="doSearch()">🔍 搜尋</button>
  </div>
  <div class="header-actions">
    <button class="btn btn-amber" onclick="toggleSummarizeBar()">📚 合成摘要</button>
  </div>
</header>

<!-- Summarize bar -->
<div class="summarize-bar" id="summarizeBar">
  <span style="font-size:.85rem;font-weight:600;color:#92400e;white-space:nowrap">合成主題摘要：</span>
  <input id="summarizeInput" type="text" placeholder="輸入主題（例如：Python、機器學習…）"
    onkeydown="if(event.key==='Enter') doSummarize()">
  <button class="btn btn-amber btn-sm" onclick="doSummarize()">產生摘要</button>
  <button class="btn btn-ghost btn-sm" onclick="toggleSummarizeBar()">✕</button>
</div>

<!-- Main layout -->
<div class="layout">

  <!-- Sidebar -->
  <aside class="sidebar">
    <div class="sidebar-top">
      <button class="btn btn-primary" onclick="openAddModal()">✏️ 新增筆記</button>
    </div>
    <div class="notes-list" id="notesList"></div>
    <div class="sidebar-bottom">
      <button class="btn btn-emerald" onclick="doMindmap()">🗺️ 知識地圖</button>
      <button class="btn btn-violet" onclick="doAnalyze()">🔬 分析知識缺口</button>
      <div class="note-count" id="noteCount"></div>
    </div>
  </aside>

  <!-- Main content -->
  <main class="main">

    <!-- Empty state -->
    <div class="empty-state" id="emptyState">
      <div class="empty-state-icon">📝</div>
      <div class="empty-state-text">點選左側筆記，或使用語意搜尋開始</div>
    </div>

    <!-- Note view -->
    <div class="note-view" id="noteView">
      <div class="note-header">
        <div class="note-title" id="noteTitle"></div>
        <div class="note-actions" id="noteActions"></div>
      </div>
      <div class="note-tags" id="noteTags"></div>
      <div class="note-meta" id="noteMeta"></div>
      <div class="note-body rich-text" id="noteBody"></div>
    </div>

    <!-- AI panel -->
    <div class="ai-panel" id="aiPanel">
      <div class="ai-panel-header">
        <div class="ai-panel-title" id="aiPanelTitle">🤖 AI 回應</div>
        <button class="btn btn-ghost btn-sm" onclick="closeAiPanel()">✕ 關閉</button>
      </div>
      <div class="ai-stream-area" id="aiStreamArea">
        <div class="ai-bubble rich-text" id="aiBubble"></div>
        <div id="mermaidContainer"></div>
        <div class="ai-results" id="aiResults"></div>
      </div>
      <div class="status-bar" id="statusBar">就緒</div>
    </div>

  </main>
</div>

<!-- Add Note Modal -->
<div class="modal-overlay" id="addModal" onclick="if(event.target===this) closeAddModal()">
  <div class="modal">
    <div class="modal-title">✏️ 新增筆記</div>
    <div class="form-group">
      <label class="form-label">標題</label>
      <input class="form-input" id="newTitle" type="text" placeholder="筆記標題…"
        onkeydown="if(event.key==='Enter') document.getElementById('newContent').focus()">
    </div>
    <div class="form-group">
      <label class="form-label">內容</label>
      <textarea class="form-input" id="newContent" placeholder="筆記內容…（支援 Markdown 格式）"></textarea>
    </div>
    <div class="modal-actions">
      <button class="btn btn-ghost" onclick="closeAddModal()">取消</button>
      <button class="btn btn-primary" id="addBtn" onclick="submitAddNote()">💾 儲存（AI 自動標籤 + 向量索引）</button>
    </div>
  </div>
</div>

<!-- Toast -->
<div class="toast" id="toast"></div>

<script>
// ── Mermaid init ────────────────────────────────────
mermaid.initialize({ startOnLoad: false, theme: 'default', securityLevel: 'loose' });

// ── State ───────────────────────────────────────────
let currentNoteId = null;
let notes = [];

// ── Init ────────────────────────────────────────────
document.addEventListener('DOMContentLoaded', async () => {
  await checkApiStatus();
  loadNotes();
});

async function checkApiStatus() {
  try {
    const res = await fetch('/api/status');
    const data = await res.json();
    if (!data.api_key_set) {
      const bar = document.createElement('div');
      bar.id = 'apiWarning';
      bar.style.cssText = 'background:#fef3c7;border-bottom:1px solid #fbbf24;padding:10px 28px;font-size:.85rem;color:#92400e;display:flex;align-items:center;gap:8px;';
      bar.innerHTML = '⚠️ <strong>尚未設定 GOOGLE_API_KEY</strong>：請在專案目錄建立 <code style="background:#fde68a;padding:1px 5px;border-radius:4px">.env</code> 檔案，內容為 <code style="background:#fde68a;padding:1px 5px;border-radius:4px">GOOGLE_API_KEY=你的金鑰</code>，再重新啟動伺服器。';
      document.querySelector('.layout').before(bar);
    }
  } catch (e) {}
}

// ── Data ────────────────────────────────────────────
async function loadNotes() {
  const res = await fetch('/api/notes');
  notes = await res.json();
  renderNoteList();
}

function renderNoteList() {
  const list = document.getElementById('notesList');
  const count = document.getElementById('noteCount');
  count.textContent = `共 ${notes.length} 篇筆記`;
  if (!notes.length) {
    list.innerHTML = '<div style="padding:20px;text-align:center;color:#64748b;font-size:.83rem">尚無筆記<br>點上方按鈕新增</div>';
    return;
  }
  list.innerHTML = notes.map(n => {
    const tagHtml = n.tags.map(t => `<span class="tag">${t}</span>`).join('');
    const date = n.updated_at.slice(0, 10);
    return `
      <div class="note-item ${n.id === currentNoteId ? 'active' : ''}"
           onclick="viewNote(${n.id})">
        <div class="note-item-title">${esc(n.title)}</div>
        <div class="note-item-meta">${date}</div>
        ${tagHtml ? `<div class="note-item-tags">${tagHtml}</div>` : ''}
      </div>`;
  }).join('');
}

// ── View note ────────────────────────────────────────
async function viewNote(id) {
  currentNoteId = id;
  closeAiPanel();
  const res = await fetch(`/api/notes/${id}`);
  if (!res.ok) { showToast('找不到筆記', 'error'); return; }
  const note = await res.json();

  show('noteView');
  hide('emptyState');

  document.getElementById('noteTitle').textContent = note.title;
  document.getElementById('noteMeta').textContent =
    `建立：${note.created_at.slice(0,19)}　更新：${note.updated_at.slice(0,19)}`;
  document.getElementById('noteTags').innerHTML =
    note.tags.map(t => `<span class="note-tag">#${t}</span>`).join('');
  document.getElementById('noteBody').innerHTML = marked.parse(note.content);
  document.getElementById('noteActions').innerHTML = `
    <button class="btn btn-teal btn-sm" onclick="doAugment(${id})">🤖 補充知識</button>
    <button class="btn btn-violet btn-sm" onclick="doImprove(${id})">✨ 改善筆記</button>
    <button class="btn btn-slate btn-sm" onclick="doAutotag(${id})">🏷 自動標籤</button>
    <button class="btn btn-rose btn-sm" onclick="doDelete(${id})">🗑 刪除</button>
  `;
  renderNoteList();
}

// ── Add note ─────────────────────────────────────────
function openAddModal() {
  document.getElementById('newTitle').value = '';
  document.getElementById('newContent').value = '';
  document.getElementById('addModal').classList.add('open');
  setTimeout(() => document.getElementById('newTitle').focus(), 50);
}
function closeAddModal() { document.getElementById('addModal').classList.remove('open'); }

async function submitAddNote() {
  const title   = document.getElementById('newTitle').value.trim();
  const content = document.getElementById('newContent').value.trim();
  if (!title || !content) { showToast('標題和內容不能為空', 'error'); return; }

  const btn = document.getElementById('addBtn');
  btn.disabled = true;
  btn.textContent = '儲存中（AI 標籤 + 向量化）…';

  const res = await fetch('/api/notes', {
    method: 'POST',
    headers: {'Content-Type':'application/json'},
    body: JSON.stringify({title, content})
  });
  const note = await res.json();
  btn.disabled = false;
  btn.innerHTML = '💾 儲存（AI 自動標籤 + 向量索引）';
  closeAddModal();
  await loadNotes();
  viewNote(note.id);
  showToast('✓ 已新增筆記（含標籤 + 向量索引）', 'success');
}

// ── Delete ───────────────────────────────────────────
async function doDelete(id) {
  const note = notes.find(n => n.id === id);
  if (!confirm(`確定刪除「${note?.title}」？此操作無法復原。`)) return;
  await fetch(`/api/notes/${id}`, {method:'DELETE'});
  currentNoteId = null;
  show('emptyState'); hide('noteView'); closeAiPanel();
  await loadNotes();
  showToast('筆記已刪除', 'success');
}

// ── Autotag ──────────────────────────────────────────
async function doAutotag(id) {
  showToast('Gemini 標籤產生中…');
  const res = await fetch(`/api/notes/${id}/autotag`, {method:'POST'});
  const data = await res.json();
  await loadNotes();
  if (currentNoteId === id) viewNote(id);
  showToast(`✓ 標籤：${data.tags.join(', ')}`, 'success');
}

// ── Search ───────────────────────────────────────────
function doSearch() {
  const query = document.getElementById('searchInput').value.trim();
  if (!query) return;
  showAiPanel(`🔍 語意搜尋：「${query}」`);
  streamSSE('/api/search', {method:'POST', body:{query}}, results => {
    if (!results) return;
    const el = document.getElementById('aiResults');
    el.innerHTML = results.map(r => `
      <div class="ai-result-item" onclick="viewNote(${r.id})">
        <div class="ai-result-item-title">#${r.id} ${esc(r.title)}</div>
        <div class="ai-result-item-snippet">${esc(r.snippet)}</div>
      </div>`).join('');
  });
}

// ── Augment ──────────────────────────────────────────
async function doAugment(id) {
  const note = notes.find(n => n.id === id) || await (await fetch(`/api/notes/${id}`)).json();
  showAiPanel(`🤖 AI 補充知識：${note.title}`);
  streamSSE(`/api/notes/${id}/augment`, {method:'POST'}, async () => {
    await loadNotes(); viewNote(id);
  });
}

// ── Improve ──────────────────────────────────────────
async function doImprove(id) {
  const note = notes.find(n => n.id === id) || await (await fetch(`/api/notes/${id}`)).json();
  showAiPanel(`✨ AI 改善筆記：${note.title}`);
  streamSSE(`/api/notes/${id}/improve`, {method:'POST'}, async () => {
    await loadNotes(); viewNote(id);
  });
}

// ── Analyze ──────────────────────────────────────────
function doAnalyze() {
  showAiPanel('🔬 AI 知識缺口分析');
  streamSSE('/api/analyze', {method:'GET'});
}

// ── Mind Map ─────────────────────────────────────────
function doMindmap() {
  showAiPanel('🗺️ 知識地圖（Mind Map）');
  streamSSE('/api/mindmap', {method:'GET'});
}

// ── Summarize ────────────────────────────────────────
function toggleSummarizeBar() {
  const bar = document.getElementById('summarizeBar');
  bar.classList.toggle('visible');
  if (bar.classList.contains('visible'))
    setTimeout(() => document.getElementById('summarizeInput').focus(), 50);
}
function doSummarize() {
  const topic = document.getElementById('summarizeInput').value.trim();
  if (!topic) return;
  showAiPanel(`📚 知識合成：「${topic}」`);
  document.getElementById('summarizeBar').classList.remove('visible');
  streamSSE('/api/summarize', {method:'POST', body:{topic}});
}

// ── SSE streaming helper ─────────────────────────────
function streamSSE(url, opts, onDone) {
  const method = opts.method || 'GET';
  const fetchOpts = { method };
  if (opts.body) {
    fetchOpts.headers = {'Content-Type':'application/json'};
    fetchOpts.body = JSON.stringify(opts.body);
  }

  const bubble = document.getElementById('aiBubble');
  bubble.innerHTML = '';
  document.getElementById('aiResults').innerHTML = '';
  document.getElementById('mermaidContainer').innerHTML = '';
  setStatus(true, 'Gemini 思考中…');

  let rawText = '';

  fetch(url, fetchOpts).then(res => {
    const reader = res.body.getReader();
    const decoder = new TextDecoder();
    let buffer = '';

    function pump() {
      reader.read().then(({ done, value }) => {
        if (done) { setStatus(false, '完成'); return; }
        buffer += decoder.decode(value, { stream: true });
        const lines = buffer.split('\\n');
        buffer = lines.pop();
        for (const line of lines) {
          if (!line.startsWith('data: ')) continue;
          try {
            const evt = JSON.parse(line.slice(6));
            if (evt.type === 'text') {
              rawText += evt.content;
              bubble.innerHTML = marked.parse(rawText);
              const area = document.getElementById('aiStreamArea');
              area.scrollTop = area.scrollHeight;
            } else if (evt.type === 'mermaid') {
              renderMermaid(evt.content);
            } else if (evt.type === 'done') {
              setStatus(false, evt.message || '完成');
              if (evt.results) onDone && onDone(evt.results);
              else onDone && onDone(null);
              if (evt.message) showToast('✓ ' + evt.message, 'success');
            } else if (evt.type === 'error') {
              setStatus(false, '錯誤：' + evt.message);
              bubble.innerHTML += `<p style="color:#e11d48">⚠️ ${esc(evt.message)}</p>`;
              showToast(evt.message, 'error');
            }
          } catch (e) {}
        }
        pump();
      });
    }
    pump();
  }).catch(e => {
    setStatus(false, '連線失敗');
    showToast('無法連線到伺服器', 'error');
  });
}

// ── Mermaid rendering ────────────────────────────────
async function renderMermaid(code) {
  const container = document.getElementById('mermaidContainer');
  try {
    const id = 'mm-' + Date.now();
    const { svg } = await mermaid.render(id, code);
    container.innerHTML = `<div class="mermaid-container">${svg}</div>`;
    const area = document.getElementById('aiStreamArea');
    area.scrollTop = area.scrollHeight;
  } catch (e) {
    // Fallback: show raw code
    container.innerHTML = `<div class="mermaid-container"><pre style="background:#f8fafc;color:#334155;padding:16px;border-radius:8px;white-space:pre-wrap">${esc(code)}</pre></div>`;
  }
}

// ── AI panel helpers ─────────────────────────────────
function showAiPanel(title) {
  document.getElementById('aiPanelTitle').textContent = title;
  hide('noteView'); hide('emptyState'); show('aiPanel');
}
function closeAiPanel() {
  hide('aiPanel');
  if (currentNoteId) show('noteView');
  else show('emptyState');
}

// ── Status bar ───────────────────────────────────────
function setStatus(loading, text) {
  const bar = document.getElementById('statusBar');
  bar.innerHTML = loading
    ? `<div class="spinner"></div><span>${text}</span>`
    : `<span>${text}</span>`;
}

// ── Utilities ────────────────────────────────────────
function show(id) { document.getElementById(id).classList.add('visible'); }
function hide(id) { document.getElementById(id).classList.remove('visible'); }
function esc(s) {
  return String(s).replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');
}

let toastTimer;
function showToast(msg, type='') {
  const t = document.getElementById('toast');
  t.textContent = msg;
  t.className = `toast ${type} show`;
  clearTimeout(toastTimer);
  toastTimer = setTimeout(() => t.classList.remove('show'), 3000);
}

document.addEventListener('keydown', e => { if (e.key === 'Escape') closeAddModal(); });
</script>
</body>
</html>
"""

with open("/content/templates/index.html", "w", encoding="utf-8") as f:
    f.write(html_code)
print("✅ index.html 寫入完成")


In [ ]:
# ⑥ 啟動 Flask 伺服器
import sys, os, threading, time

os.chdir('/content')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# 清除舊的模組快取（若重新執行此儲存格）
for mod in list(sys.modules.keys()):
    if 'app' in mod and 'google' not in mod and 'colab' not in mod:
        del sys.modules[mod]

from app import app, init_db
init_db()

flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, debug=False,
                           use_reloader=False, threaded=True),
    daemon=True
)
flask_thread.start()
time.sleep(2)

# 確認伺服器啟動
import urllib.request
try:
    urllib.request.urlopen('http://localhost:5000/api/status', timeout=5)
    print("✅ Flask 伺服器已啟動（port 5000）")
except Exception as e:
    print(f"⚠️  伺服器啟動確認失敗：{e}，但可能仍在啟動中...")


In [ ]:
# ⑦ 建立公開網址（Cloudflare Tunnel）並開啟應用程式
import subprocess, threading, re, time
from IPython.display import display, HTML

public_url = None

def start_tunnel():
    global public_url
    # 下載 cloudflared
    dl = subprocess.run(
        ['wget', '-q', '-O', '/usr/local/bin/cloudflared',
         'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
        capture_output=True
    )
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])

    proc = subprocess.Popen(
        ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        universal_newlines=True, bufsize=1
    )

    for line in proc.stdout:
        line = line.strip()
        if line:
            print(f"  cloudflared: {line}")
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group()
            break

t = threading.Thread(target=start_tunnel, daemon=True)
t.start()

print("⏳ 建立公開網址中（約 10-20 秒）...")
for i in range(30):
    if public_url:
        break
    time.sleep(1)

if public_url:
    display(HTML(f'''
    <div style="background:#d1fae5;border:1px solid #6ee7b7;border-radius:12px;padding:20px 24px;margin:12px 0;font-family:sans-serif">
      <div style="font-size:1.1rem;font-weight:700;color:#065f46;margin-bottom:8px">🎉 智腦記事本已啟動！</div>
      <div style="margin-bottom:12px;color:#047857">點擊下方網址開啟應用程式：</div>
      <a href="{public_url}" target="_blank"
         style="display:inline-block;background:#059669;color:#fff;padding:10px 24px;
                border-radius:8px;text-decoration:none;font-size:1rem;font-weight:600">
        🌐 開啟 智腦記事本 →
      </a>
      <div style="margin-top:12px;font-size:.85rem;color:#6b7280">
        網址：{public_url}<br>
        ⚠️ 此網址為臨時網址，重新執行本儲存格會產生新網址
      </div>
    </div>
    '''))
else:
    print("⚠️  無法取得公開網址，請確認網路連線後重新執行")
    print("   備用方案：可使用 ngrok（見下方備用儲存格）")


In [ ]:
# ⑧ 備用方案：使用 ngrok（若 Cloudflare Tunnel 失敗）
# 需要先到 https://ngrok.com 免費註冊，取得 Authtoken

# NGROK_TOKEN = "你的_ngrok_authtoken"  # ← 填入後取消此行與下方的註解

# !pip install -q pyngrok
# from pyngrok import ngrok, conf
# conf.get_default().auth_token = NGROK_TOKEN
# tunnel = ngrok.connect(5000)
# print(f"🌐 ngrok 網址：{tunnel.public_url}")
